# 02 — Limpieza y feature engineering
### Proyecto: Predicción de duración de viajes — NYC Yellow Taxi (ene-abr 2026)

**Avance 1.** Este notebook documenta las decisiones de limpieza y de construcción de variables
que usa el pipeline (`src/data.py`, `src/features.py`). Todo lo que se decide aquí se aplica
igual en `src/train.py`, para que el notebook y el script no se desincronicen.

Contenido:
1. Carga por mes con muestreo reproducible
2. Reglas de limpieza y cuántos registros elimina cada una
3. Faltantes estructurales y cómo se tratan
4. Variables nuevas (todas disponibles al momento del pickup)
5. Control de leakage en codificaciones que usan el target
6. Deriva entre meses (por qué el split es temporal)

In [ ]:
import os, sys
sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import data as D
from features import add_features, ALL_FEATURES, NUMERIC_FEATURES, ONEHOT_FEATURES, TARGET_ENC_FEATURES, TARGET

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
SEED = 42
np.random.seed(SEED)

# Ruta al parquet combinado (ver README). Se puede cambiar con la variable de entorno TAXI_DATA.
DATA_PATH = os.environ.get('TAXI_DATA', '../data/yellow_2026_jan_apr.parquet')
FIG = '../reports/figures'
os.makedirs(FIG, exist_ok=True)

## 1. Carga

Se carga **un mes a la vez** y se muestrea en el momento (`load_months`), así el pico de memoria es
de ~1 mes y no de 4. La muestra es aleatoria simple dentro de cada mes, con semilla fija.

In [ ]:
N_POR_MES = 300_000   # suficiente para describir distribuciones; el modelo usa más filas en train.py
raw = D.load_months(DATA_PATH, n_per_month=N_POR_MES, seed=SEED)
raw = D.add_target_and_time_features(raw)
print(f'{len(raw):,} filas cargadas')
raw['source_month'].value_counts().sort_index()

## 2. Reglas de limpieza

| Regla | Justificación |
|---|---|
| pickup fuera del mes del archivo | El TLC publica algunos registros con fechas de otros años (relojes mal configurados). Además, rompen el split temporal: un viaje "de 2009" dentro del archivo de abril no es un viaje de abril. |
| duración < 1 min | Incluye las duraciones negativas (~1.24%) y viajes de segundos con distancia > 0: cancelaciones o errores del taxímetro. |
| duración > 180 min | 0.04%; casi siempre dropoffs que el conductor no cerró. |
| distancia ≤ 0 | ~3.1%; sin distancia el viaje no tiene sentido físico para esta tarea. |
| distancia > 100 mi | Máximo observado 328,522 mi: error del taxímetro. |
| velocidad media > 80 mph | Combinación distancia/duración imposible en NYC (aunque cada variable por separado parezca normal). |

Las reglas usan la duración (el target) para detectar **etiquetas corruptas**. Se aplican igual a train,
validación y test. La consecuencia, que se reporta como limitación, es que las métricas describen el
desempeño sobre viajes con registro válido, no sobre todos los registros que llegan.

In [ ]:
clean, reporte = D.clean_for_modeling(raw, return_report=True)
reporte

In [ ]:
# Cómo se ven los registros eliminados por la regla de velocidad (casos sospechosos)
speed = raw['trip_distance'] / (raw['duration_min'].clip(lower=1e-3) / 60)
sosp = raw.loc[(speed > 80) & (raw['duration_min'] >= 1), ['tpep_pickup_datetime', 'duration_min', 'trip_distance', 'PULocationID', 'DOLocationID', 'VendorID']]
print(f'{len(sosp):,} registros con velocidad media > 80 mph (y duración >= 1 min)')
sosp.head(10)

## 3. Faltantes estructurales

En la entrega previa vimos que `RatecodeID`, `passenger_count` y `store_and_fwd_flag` faltan juntos en ~26%
de los registros y que se concentran en ciertos `VendorID`. No es faltante al azar (MCAR), así que:

- **No se eliminan** esas filas (sería sesgar la muestra hacia ciertos proveedores).
- `RatecodeID` faltante pasa a ser **una categoría propia** (`-1`) + un indicador `ratecode_missing`.
- `passenger_count` se imputa con la mediana **dentro del Pipeline** (la mediana se calcula solo con train).

In [ ]:
falt = clean.groupby('VendorID').agg(
    n=('RatecodeID', 'size'),
    pct_ratecode_faltante=('RatecodeID', lambda s: round(100 * s.isna().mean(), 1)),
    pct_pasajeros_faltante=('passenger_count', lambda s: round(100 * s.isna().mean(), 1)),
    duracion_mediana=('duration_min', 'median'),
)
falt

In [ ]:
# ¿Los viajes con RatecodeID faltante duran distinto? Si sí, el indicador aporta señal.
fig, ax = plt.subplots(figsize=(8, 4))
for flag, g in clean.groupby(clean['RatecodeID'].isna()):
    ax.hist(g['duration_min'].clip(upper=60), bins=60, density=True, histtype='step', lw=1.5,
            label='RatecodeID faltante' if flag else 'RatecodeID presente')
ax.set_xlabel('Duración (min, recortada a 60)'); ax.set_ylabel('densidad'); ax.legend()
ax.set_title('Duración según faltante de RatecodeID')
plt.savefig(f'{FIG}/04_duracion_por_faltante.png', dpi=130, bbox_inches='tight'); plt.show()

## 4. Variables nuevas

Todas son funciones deterministas de columnas conocidas al pickup y no dependen de otras filas,
así que se pueden calcular antes del split sin riesgo de leakage:

| Variable | Qué captura |
|---|---|
| `log_distance` | La relación distancia-duración no es lineal (viajes largos van por autopista, más rápido). |
| `hour_sin`, `hour_cos` | Hora como variable cíclica (23:59 está cerca de 00:00). |
| `is_weekend`, `is_holiday` | Fin de semana y feriados federales de EE.UU. (calendario de pandas, sin fuente externa). |
| `same_zone` | Viajes dentro de la misma zona (cortos, muy ruidosos). |
| `pu_airport`, `do_airport` | Zonas TLC 1 (EWR), 132 (JFK), 138 (LGA). |
| `unknown_zone` | Zonas 264/265 ("Unknown"/"Outside NYC"). |
| `route` | Par origen-destino (≈70k combinaciones posibles). |

**Corrección respecto a la entrega previa:** el baseline lineal usaba `PULocationID` y `DOLocationID`
como números. Eso le dice al modelo que la zona 200 es "el doble" que la 100, lo cual no tiene sentido.
Ahora las zonas y la ruta se codifican con *target encoding*, y el baseline lineal usa solo
distancia + hora y día en one-hot.

In [ ]:
feat = add_features(clean)
feat[ALL_FEATURES].describe(include='all').T

In [ ]:
# Duración mediana: viajes de/hacia aeropuerto y dentro de la misma zona
pd.DataFrame({
    'desde aeropuerto': feat.loc[feat.pu_airport == 1, 'duration_min'].median(),
    'hacia aeropuerto': feat.loc[feat.do_airport == 1, 'duration_min'].median(),
    'misma zona': feat.loc[feat.same_zone == 1, 'duration_min'].median(),
    'feriado': feat.loc[feat.is_holiday == 1, 'duration_min'].median(),
    'resto': feat.loc[(feat.pu_airport == 0) & (feat.do_airport == 0) & (feat.same_zone == 0), 'duration_min'].median(),
}, index=['duración mediana (min)']).T.round(2)

In [ ]:
# La velocidad media cambia con la distancia -> justifica log_distance y modelos no lineales
feat['vel_mph'] = feat['trip_distance'] / (feat['duration_min'] / 60)
feat['dist_bin'] = pd.cut(feat['trip_distance'], [0, 1, 2, 3, 5, 10, 20, 100])
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=feat, x='dist_bin', y='vel_mph', showfliers=False, ax=ax, color='#93c5fd')
ax.set_xlabel('Distancia (mi)'); ax.set_ylabel('Velocidad media (mph)')
ax.set_title('Velocidad media por distancia del viaje')
plt.savefig(f'{FIG}/05_velocidad_por_distancia.png', dpi=130, bbox_inches='tight'); plt.show()

## 5. Leakage en las codificaciones con target

`PULocationID`, `DOLocationID` y `route` se codifican con `TargetEncoder` (media de la duración por categoría).
Esto podría filtrar el target si se hace mal. Controles:

1. El encoder vive **dentro del `Pipeline`**: se ajusta con `fit` sobre train y en validación/test solo se aplica `transform`.
2. Al transformar el propio train usa **cross-fitting de 5 folds**: cada fila recibe la media calculada sin ella.
3. Las rutas que no aparecen en train reciben la media global (no información del futuro).

Además, sigue vigente el riesgo de `trip_distance` (distancia real, no estimada). En `train.py` se corre un
**experimento de sensibilidad**: el mismo modelo sin distancia, para medir cuánto depende de ella.

## 6. Deriva entre meses

Si la distribución del target cambia entre meses, un split aleatorio daría métricas optimistas.
Comparamos la duración por mes:

In [ ]:
deriva = feat.groupby('source_month')['duration_min'].describe(percentiles=[.25, .5, .75, .95]).round(2)
deriva

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for m, g in feat.groupby('source_month'):
    ax.hist(g['duration_min'].clip(upper=90), bins=90, histtype='step', density=True, lw=1.4, label=m)
ax.set_xlabel('Duración (min, recortada a 90)'); ax.set_ylabel('densidad'); ax.legend()
ax.set_title('Distribución del target por mes')
plt.savefig(f'{FIG}/06_deriva_target_por_mes.png', dpi=130, bbox_inches='tight'); plt.show()

**Lectura:** escribir aquí si la mediana y los percentiles altos se mueven entre meses (p. ej. invierno vs. primavera,
feriados de enero/febrero). Cualquier diferencia refuerza la decisión de validar en un mes que el modelo no vio.

**Decisión de partición (avance 1):** train = enero-febrero, validación = marzo (para elegir modelo),
test = abril (se usa una sola vez al final).